# Data Pipeline with `create_dataset`

This notebook demonstrates the `create_dataset` function, which builds a
`tf.data.Dataset` pipeline for loading `.npy` density fields, applying
normalisation, optional masking, and batching.

We cover:

- 1-channel mode (no mask)
- 1-channel mode with a mask (observed * mask)
- 2-channel mode (observed * mask concatenated with the mask)
- Options: `shuffle`, `repeat`, `drop_remainder`
- Visualising a batch

In [ ]:
import os, tempfile, glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from CosmoRecon import create_dataset

## Generate synthetic data

In [ ]:
FIELD_SIZE = 16
N_FILES = 8
NORM_VAL = 1.0

tmp = tempfile.mkdtemp(prefix="cosmorecon_pipeline_")
obs_dir  = os.path.join(tmp, "observed")
true_dir = os.path.join(tmp, "true")
os.makedirs(obs_dir); os.makedirs(true_dir)

rng = np.random.default_rng(0)
for i in range(N_FILES):
    np.save(os.path.join(obs_dir,  f"f_{i:02d}.npy"), rng.random((FIELD_SIZE,)*3, dtype=np.float32))
    np.save(os.path.join(true_dir, f"f_{i:02d}.npy"), rng.random((FIELD_SIZE,)*3, dtype=np.float32))

# Binary mask: 1 = observed, 0 = missing
mask = np.ones((FIELD_SIZE,)*3, dtype=np.float32)
mask[4:12, 4:12, 4:12] = 0.0

x_files = sorted(glob.glob(os.path.join(obs_dir,  "*.npy")))
y_files = sorted(glob.glob(os.path.join(true_dir, "*.npy")))
print(f"{N_FILES} file pairs ready, field_size={FIELD_SIZE}")

## 1-channel, no mask

The simplest mode: each sample yields `(obs, true)` tensors with shape
`(N, N, N, 1)`, normalised by `norm_val`.

In [ ]:
ds_plain = create_dataset(
    x_files, y_files,
    batch_size=4,
    field_size=FIELD_SIZE,
    norm_val=NORM_VAL,
    shuffle=False,
)

for xb, yb in ds_plain.take(1):
    print(f"x shape: {xb.shape}  y shape: {yb.shape}")
    print(f"x range: [{xb.numpy().min():.4f}, {xb.numpy().max():.4f}]")

## 1-channel with mask

When a mask is provided with `channels=1`, each observed field is
multiplied element-wise by the mask: `x = obs * mask`.
Missing voxels become zero.

In [ ]:
ds_masked_1ch = create_dataset(
    x_files, y_files,
    batch_size=4,
    field_size=FIELD_SIZE,
    norm_val=NORM_VAL,
    mask=mask,
    channels=1,
    shuffle=False,
)

for xb, yb in ds_masked_1ch.take(1):
    print(f"x shape: {xb.shape}  (channels=1, with mask applied)")
    # The missing region should be exactly 0
    mid = FIELD_SIZE // 2
    hole_vals = xb[0, 4:12, 4:12, mid, 0].numpy()
    print(f"Values inside the hole (should be 0): min={hole_vals.min()}, max={hole_vals.max()}")

## 2-channel with mask

With `channels=2`, the dataset concatenates the masked observed field and
the mask along the last axis:  
`x = concat(obs * mask, mask)` → shape `(N, N, N, 2)`.

This lets the network "see" which voxels are missing.

In [ ]:
ds_masked_2ch = create_dataset(
    x_files, y_files,
    batch_size=4,
    field_size=FIELD_SIZE,
    norm_val=NORM_VAL,
    mask=mask,
    channels=2,
    shuffle=False,
)

for xb, yb in ds_masked_2ch.take(1):
    print(f"x shape: {xb.shape}  (channel 0 = masked field, channel 1 = mask)")
    print(f"y shape: {yb.shape}")

    # Channel 1 is the mask: should be binary
    mask_ch = xb[0, :, :, :, 1].numpy()
    print(f"Mask channel unique values: {np.unique(mask_ch)}")

## Effect of `norm_val`

All loaded values are divided by `norm_val`.  For real cosmological
data a typical value is 40 (the approximate maximum density).

In [ ]:
for nv in [1.0, 10.0, 40.0]:
    ds = create_dataset(
        x_files, y_files,
        batch_size=N_FILES,
        field_size=FIELD_SIZE,
        norm_val=nv,
        shuffle=False,
    )
    for xb, _ in ds.take(1):
        vals = xb.numpy()
        print(f"norm_val={nv:5.1f}  ->  x range: [{vals.min():.4f}, {vals.max():.4f}]")

## `drop_remainder` and `repeat`

- `drop_remainder=True` drops the last batch if it is smaller than `batch_size`.
- `repeat=True` loops the dataset indefinitely (useful with `steps_per_epoch`).

In [ ]:
# 8 files with batch_size=3 -> 2 full batches + 1 partial (size 2)
ds_nodrop = create_dataset(x_files, y_files, batch_size=3, field_size=FIELD_SIZE,
                           norm_val=NORM_VAL, shuffle=False)
batch_sizes_nodrop = [xb.shape[0] for xb, _ in ds_nodrop]
print(f"drop_remainder=False: batch sizes = {batch_sizes_nodrop}")

ds_drop = create_dataset(x_files, y_files, batch_size=3, field_size=FIELD_SIZE,
                         norm_val=NORM_VAL, shuffle=False, drop_remainder=True)
batch_sizes_drop = [xb.shape[0] for xb, _ in ds_drop]
print(f"drop_remainder=True:  batch sizes = {batch_sizes_drop}")

# repeat + take
ds_rep = create_dataset(x_files, y_files, batch_size=4, field_size=FIELD_SIZE,
                        norm_val=NORM_VAL, shuffle=False, repeat=True)
n_batches = 0
for _ in ds_rep.take(10):
    n_batches += 1
print(f"repeat=True, take(10): got {n_batches} batches (dataset repeats indefinitely)")

## Visualise a batch

Central z-slice for the first four samples in a 2-channel masked dataset.

In [ ]:
for xb, yb in ds_masked_2ch.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    mid = FIELD_SIZE // 2

    for i in range(4):
        axes[0, i].imshow(xb[i, :, :, mid, 0].numpy(), origin="lower", cmap="inferno")
        axes[0, i].set_title(f"Observed (masked) #{i}")
        axes[1, i].imshow(yb[i, :, :, mid, 0].numpy(), origin="lower", cmap="inferno")
        axes[1, i].set_title(f"Target #{i}")

    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle(f"Central z-slice (z={mid}), 2-channel dataset")
    plt.tight_layout()
    plt.show()